# Ollama Colab

Run a large language model on a free Google Colab GPU and expose it as a public API endpoint.

| Feature | Details |
|---|---|
| Model selection | Browse & search the live Ollama library — pick any model |
| Tunnel options | Cloudflare quick tunnel *(default)*, Cloudflare named tunnel, ngrok |

**Setup**: `Runtime → Change runtime type → T4 GPU → Save`, then run cells in order.

## Step 1: Install Dependencies

In [ ]:
# Verify GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

# Install system utilities
!sudo apt install zstd lshw

import torch
from psutil import virtual_memory

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"\nGPU : {gpu_name}  ({gpu_mem:.1f} GB VRAM)")
else:
    print("\nERROR: No GPU detected.")
    print("Go to Runtime → Change runtime type → T4 GPU, then re-run.")
    raise SystemExit("No GPU — stopping.")

ram_gb = virtual_memory().total / 1e9
print(f"RAM : {ram_gb:.1f} GB  ({'high-RAM runtime' if ram_gb >= 20 else 'standard runtime'})")

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install cloudflared (.deb — system-wide)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb \
    -O /tmp/cloudflared.deb
!dpkg -i /tmp/cloudflared.deb 2>/dev/null || apt-get install -y /tmp/cloudflared.deb 2>/dev/null

# Install Python dependencies
!pip install pyngrok requests beautifulsoup4 ipywidgets -q

print("\nAll dependencies installed.")

## Step 2: Choose Model

Browse the live Ollama model library, filter by category, search by name, then pick a tag to set your `MODEL_NAME`.

In [ ]:
import requests
from bs4 import BeautifulSoup
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Scrape Ollama library ────────────────────────────────────────────────────
def fetch_ollama_library():
    """Scrape ollama.com/library and return a list of model dicts."""
    print("Fetching model list from ollama.com/library ...")
    resp = requests.get("https://ollama.com/library", timeout=15,
                        headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    models = []
    for a in soup.select("ul li a[href^='/library/']"):
        name_el   = a.select_one("h2, [class*='truncate']")
        desc_el   = a.select_one("p, [class*='description']")
        pulls_el  = a.find(string=lambda t: t and "Pulls" in t)
        name = name_el.get_text(strip=True) if name_el else a["href"].split("/")[-1]
        desc = desc_el.get_text(strip=True) if desc_el else ""
        pulls = pulls_el.strip() if pulls_el else ""

        # Collect size/capability tags (small pill-style spans)
        tags = [s.get_text(strip=True) for s in a.select("span, li")
                if s.get_text(strip=True) and len(s.get_text(strip=True)) < 20
                and s.get_text(strip=True) not in (name, desc, pulls)]
        # Deduplicate while preserving order
        seen = set(); unique_tags = []
        for t in tags:
            if t not in seen and t not in (pulls,):
                seen.add(t); unique_tags.append(t)

        models.append({
            "name": name,
            "desc": desc,
            "pulls": pulls,
            "tags": unique_tags,
        })

    # Fallback: parse from plain-text link structure if CSS selectors missed items
    if len(models) < 5:
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if href.startswith("/library/") and href.count("/") == 2:
                name = href.split("/")[-1]
                if not any(m["name"] == name for m in models):
                    models.append({"name": name, "desc": "", "pulls": "", "tags": []})

    print(f"Found {len(models)} models.")
    return models

ALL_MODELS = fetch_ollama_library()

# ── Derive size tags for each model ──────────────────────────────────────────
SIZE_SUFFIXES = ("b", "m", "k")
def size_tags(model):
    return [t for t in model["tags"]
            if t.lower().replace(".","").rstrip("bmk").isdigit() or
               any(t.lower().endswith(s) for s in SIZE_SUFFIXES)]

# ── Capability categories ─────────────────────────────────────────────────────
CAPABILITIES = ["all", "tools", "vision", "thinking", "embedding", "code", "cloud"]
CAP_KEYWORDS  = {
    "tools":     ["tools"],
    "vision":    ["vision"],
    "thinking":  ["thinking"],
    "embedding": ["embedding", "embed"],
    "code":      ["code", "coder", "coding", "starcoder", "deepseek-coder", "codellama"],
    "cloud":     ["cloud"],
}

def model_matches_cap(model, cap):
    if cap == "all":
        return True
    kws = CAP_KEYWORDS[cap]
    combined = (model["name"] + " " + " ".join(model["tags"])).lower()
    return any(k in combined for k in kws)

# ── Widgets ──────────────────────────────────────────────────────────────────
_style  = {"description_width": "80px"}
_layout = widgets.Layout(width="100%")

w_search = widgets.Text(
    placeholder="Search models...", description="Search:", style=_style, layout=_layout)
w_cap = widgets.ToggleButtons(
    options=CAPABILITIES, value="all", description="Filter:",
    style={"button_width": "90px", "description_width": "80px"},
    layout=_layout)
w_list = widgets.Select(
    options=[], rows=12, description="Model:", style=_style,
    layout=widgets.Layout(width="100%"))
w_tag_label = widgets.HTML("<b>Tag / variant:</b>")
w_tags = widgets.RadioButtons(options=[], description="", layout=_layout)
w_info  = widgets.HTML("")
w_apply = widgets.Button(description="✅  Use this model",
                         button_style="success",
                         layout=widgets.Layout(width="220px"))
w_out   = widgets.Output()

# ── Tag fetcher ───────────────────────────────────────────────────────────────
_tag_cache = {}

def fetch_tags(model_name):
    if model_name in _tag_cache:
        return _tag_cache[model_name]
    try:
        resp = requests.get(f"https://ollama.com/library/{model_name}/tags",
                            timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        tag_els = soup.select("a[href*='/library/'][href*=':']")
        tags = []
        for el in tag_els:
            tag = el["href"].split(":")[-1].split("/")[0].strip()
            if tag and ":" not in tag and tag not in tags:
                tags.append(tag)
        # Fallback: look for text that looks like a tag
        if not tags:
            for el in soup.find_all(string=True):
                t = el.strip()
                if t.startswith(model_name + ":"):
                    tag = t.split(":")[1].strip()
                    if tag and tag not in tags:
                        tags.append(tag)
        if not tags:
            tags = ["latest"]
        _tag_cache[model_name] = tags
        return tags
    except Exception:
        _tag_cache[model_name] = ["latest"]
        return ["latest"]

# ── Refresh model list ────────────────────────────────────────────────────────
def refresh_list(*_):
    query = w_search.value.strip().lower()
    cap   = w_cap.value
    filtered = [m for m in ALL_MODELS
                if model_matches_cap(m, cap)
                and (not query or query in m["name"].lower() or query in m["desc"].lower())]
    labels = []
    for m in filtered:
        sizes = [t for t in size_tags(m)][:4]
        size_str = "  " + " ".join(sizes) if sizes else ""
        pulls = m["pulls"].replace(" Pulls", "").strip() if m["pulls"] else ""
        pulls_str = f"  [{pulls}]" if pulls else ""
        labels.append(f"{m['name']}{size_str}{pulls_str}")
    w_list.options = labels
    if labels:
        w_list.value = labels[0]

# ── Update tag radio buttons when model selected ──────────────────────────────
def on_model_select(change):
    if not w_list.value:
        return
    model_name = w_list.value.split()[0]
    w_info.value = "<i>Loading tags...</i>"
    w_tags.options = []
    tags = fetch_tags(model_name)
    # Show the most useful tags first (latest, then sorted)
    ordered = sorted(tags, key=lambda t: (t != "latest", t))[:20]
    w_tags.options = ordered
    if ordered:
        w_tags.value = ordered[0]
    # Show model description
    meta = next((m for m in ALL_MODELS if m["name"] == model_name), None)
    desc = meta["desc"] if meta else ""
    pulls = meta["pulls"] if meta else ""
    w_info.value = (
        f"<b>{model_name}</b>"
        + (f" — {desc}" if desc else "")
        + (f"<br><small>{pulls}</small>" if pulls else "")
    )

# ── Apply button ──────────────────────────────────────────────────────────────
MODEL_NAME = None   # will be set when user clicks Apply

def on_apply(_):
    global MODEL_NAME
    if not w_list.value or not w_tags.value:
        return
    model_base = w_list.value.split()[0]
    tag        = w_tags.value
    MODEL_NAME = f"{model_base}:{tag}"
    with w_out:
        clear_output()
        print(f"✅  MODEL_NAME set to: {MODEL_NAME}")
        print("Continue to Step 3 when ready.")

w_search.observe(refresh_list, names="value")
w_cap.observe(refresh_list, names="value")
w_list.observe(on_model_select, names="value")
w_apply.on_click(on_apply)

# ── Initial render ────────────────────────────────────────────────────────────
refresh_list()

display(widgets.VBox([
    widgets.HTML("<h4>🦙 Ollama Model Browser</h4>"),
    w_search,
    w_cap,
    w_list,
    w_tag_label,
    w_tags,
    w_info,
    w_apply,
    w_out,
], layout=widgets.Layout(padding="8px", border="1px solid #ccc", border_radius="6px")))

## Step 3: Configure Environment & Tunnel

Use the dropdown to choose your tunnel. **Default is Cloudflare quick tunnel** (no credentials needed).

To use a stable URL, add a secret in Colab's key icon sidebar:

| Secret name | Where to get it |
|---|---|
| `cloudflare_token` | [Cloudflare dashboard](https://one.dash.cloudflare.com) → Networks → Tunnels → Create tunnel |
| `ngrok_authtoken` | [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken) |

In [ ]:
# @title Tunnel configuration { run: "auto" }
TUNNEL_METHOD = "Cloudflare quick tunnel (no credentials)"  # @param ["Cloudflare quick tunnel (no credentials)", "Cloudflare named tunnel (token)", "ngrok (token)"]

import os
import re
import json
import subprocess
import threading
import time
import requests
from google.colab import userdata

os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia'

# ── Verify model was selected in Step 2 ─────────────────────────────────────
if not MODEL_NAME:
    raise RuntimeError(
        "MODEL_NAME is not set.\n"
        "Go back to Step 2, choose a model and tag, then click '✅ Use this model'."
    )

# ── Environment variables ────────────────────────────────────────────────────
os.environ['OLLAMA_KEEP_ALIVE']      = '-1'
os.environ['OLLAMA_CONTEXT_LENGTH']  = '16384'
os.environ['OLLAMA_HOST']            = '0.0.0.0'

# ── Map dropdown to internal tunnel type ─────────────────────────────────────
_method_map = {
    "Cloudflare quick tunnel (no credentials)": "quick",
    "Cloudflare named tunnel (token)":          "cloudflare",
    "ngrok (token)":                            "ngrok",
}
tunnel_type  = _method_map[TUNNEL_METHOD]
tunnel_token = None

if tunnel_type == "cloudflare":
    try:
        tunnel_token = userdata.get("cloudflare_token")
        if not tunnel_token:
            raise ValueError("Secret 'cloudflare_token' is empty.")
        print("Cloudflare named tunnel — token loaded.")
    except Exception as e:
        print(f"ERROR: {e}")
        print("Add 'cloudflare_token' to Colab Secrets, or switch to the quick tunnel option.")
        raise

elif tunnel_type == "ngrok":
    try:
        tunnel_token = userdata.get("ngrok_authtoken")
        if not tunnel_token:
            raise ValueError("Secret 'ngrok_authtoken' is empty.")
        print("ngrok — token loaded.")
        print("NOTE: ngrok free-tier URLs rotate every ~2 hours.")
    except Exception as e:
        print(f"ERROR: {e}")
        print("Add 'ngrok_authtoken' to Colab Secrets, or switch to the quick tunnel option.")
        raise
else:
    print("Cloudflare quick tunnel — no credentials needed.")
    print("NOTE: The public URL will change every time you run Step 6.")

# ── Process registry ─────────────────────────────────────────────────────────
_bg_processes = {}

print(f"\nModel  : {MODEL_NAME}")
print(f"Tunnel : {TUNNEL_METHOD}")

## Step 4: Helper Functions

In [ ]:
def stream_output(process, label=""):
    """Read process stdout in a background thread and print each line."""
    prefix = f"[{label}] " if label else ""
    for line in process.stdout:
        line = line.strip()
        if line:
            print(f"{prefix}{line}")


def wait_for_ollama(timeout=60):
    """Block until the Ollama HTTP API responds or timeout is reached."""
    for i in range(timeout):
        if _bg_processes.get('ollama') and _bg_processes['ollama'].poll() is not None:
            raise RuntimeError(
                f"ollama serve exited early "
                f"(code {_bg_processes['ollama'].returncode}). "
                "Check the output above — there may be a GPU driver issue."
            )
        try:
            r = requests.get("http://localhost:11434", timeout=2)
            if r.status_code in (200, 404):
                print(f"Ollama is up (after {i+1}s).")
                return
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(1)
    raise RuntimeError("Ollama did not start within 60 s.")


print("Helper functions ready.")

## Step 5: Start Ollama Server

In [ ]:
# Kill any stale process from a previous run in this session
!pkill -f 'ollama serve' 2>/dev/null || true
time.sleep(1)

print("Starting ollama serve...")
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
)
_bg_processes['ollama'] = ollama_proc

threading.Thread(
    target=stream_output, args=(ollama_proc, 'ollama'), daemon=True
).start()

wait_for_ollama()

## Step 6: Pull Model

In [ ]:
print(f"Pulling {MODEL_NAME} ...")
print("This may take 5–15 minutes depending on your connection speed.")

pull = subprocess.Popen(
    ['ollama', 'pull', MODEL_NAME],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
)
for line in pull.stdout:
    print(line.strip())

rc = pull.wait()
if rc == 0:
    print(f"\nSuccessfully pulled {MODEL_NAME}!")
else:
    raise RuntimeError(f"Model pull failed (exit code {rc}). Try restarting the runtime.")

## Step 7: Start Tunnel

In [ ]:
public_url = None

# ── Cloudflare named tunnel ──────────────────────────────────────────────────
if tunnel_type == 'cloudflare':
    print("Starting Cloudflare named tunnel...")
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', tunnel_token],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
    )
    _bg_processes['tunnel'] = cf_proc
    threading.Thread(target=stream_output, args=(cf_proc, 'cf'), daemon=True).start()

    print("Waiting for tunnel to connect (up to 30 s)...")
    for _ in range(15):
        time.sleep(2)
        if cf_proc.poll() is not None:
            raise RuntimeError(
                f"cloudflared exited early (code {cf_proc.returncode}). "
                "Check that your tunnel token is valid and the tunnel is active "
                "in the Cloudflare dashboard."
            )
        try:
            requests.get("http://localhost:11434/api/tags", timeout=3)
            print("Cloudflare Tunnel daemon is running.")
            print("Use the hostname configured in your Cloudflare dashboard as the public URL.")
            public_url = "https://<your-cloudflare-hostname>"
            break
        except Exception:
            pass

# ── Cloudflare quick tunnel ──────────────────────────────────────────────────
elif tunnel_type == 'quick':
    print("Starting Cloudflare quick tunnel (no credentials)...")
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:11434', '--no-autoupdate'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
    )
    _bg_processes['tunnel'] = cf_proc

    # Read lines until we find the trycloudflare.com URL, then hand off to a
    # background thread so the cell completes cleanly.
    print("Waiting for public URL...")
    for line in cf_proc.stdout:
        line = line.strip()
        if line:
            print(f"[cf] {line}")
        match = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if match:
            public_url = match.group(1)
            break

    # Hand remaining stdout to background thread so cell doesn't block
    threading.Thread(target=stream_output, args=(cf_proc, 'cf'), daemon=True).start()

    if not public_url:
        raise RuntimeError("Could not find a Cloudflare quick-tunnel URL in output.")

# ── ngrok ────────────────────────────────────────────────────────────────────
elif tunnel_type == 'ngrok':
    from pyngrok import ngrok as _ngrok
    _ngrok.set_auth_token(tunnel_token)
    _ngrok.kill()
    time.sleep(1)

    print("Starting ngrok tunnel...")
    tunnel = _ngrok.connect(11434, "http", options={"host_header": "localhost:11434"})
    public_url = tunnel.public_url
    print(f"ngrok tunnel open: {public_url}")

# ── Summary ──────────────────────────────────────────────────────────────────
if public_url:
    print('\n' + '='*60)
    print(f'  Ollama is live!')
    print(f'  Model  : {MODEL_NAME}')
    print(f'  URL    : {public_url}')
    print(f'  Tunnel : {tunnel_type}')
    print('='*60)
    if tunnel_type == 'quick':
        print('\nNOTE: This URL is temporary — it changes every time you run this cell.')
        print('Add a cloudflare_token or ngrok_authtoken secret for a stable URL.')
    elif tunnel_type == 'ngrok':
        print('\nNOTE: ngrok free-tier URLs rotate every ~2 hours.')
else:
    print("\nPublic URL could not be determined. Check the output above.")

## Step 8: Test Endpoint

In [ ]:
# @title { run: "auto" }
TEST_PROMPT = "Write a hello world program in Python with a comment on each line."  # @param {type:"string"}

if not public_url or public_url.startswith("https://<your"):
    print("Set public_url to your actual tunnel hostname before running this cell.")
else:
    # Confirm endpoint is reachable
    try:
        r = requests.get(f"{public_url}/api/tags", timeout=10)
        models = [m['name'] for m in r.json().get('models', [])]
        print(f"Endpoint reachable. Available models: {models}")
    except Exception as e:
        print(f"Endpoint not reachable yet: {e}")
        print("Wait a few seconds and re-run this cell.")
    else:
        print(f"\nSending test prompt to {MODEL_NAME}...\n")
        try:
            resp = requests.post(
                f"{public_url}/api/chat",
                json={
                    "model":    MODEL_NAME,
                    "messages": [{"role": "user", "content": TEST_PROMPT}],
                    "stream":   False,
                },
                timeout=120,
            )
            resp.raise_for_status()
            print(resp.json()["message"]["content"])
        except Exception as e:
            print(f"Chat request failed: {e}")

## Step 9: Keep Server Running

Run this cell to keep the server alive. It blocks until you stop it (■) or the session disconnects.

Stopping this cell will **terminate Ollama and the tunnel**.

In [ ]:
if not public_url:
    print("Server not started. Run previous cells first.")
else:
    print(f"Server running at : {public_url}")
    print(f"Model             : {MODEL_NAME}")
    print(f"Tunnel            : {tunnel_type}")
    print(f"Context window    : 16384 tokens")
    print("\nKeeping server alive. Interrupt this cell (■) to shut down.")

    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("\nShutting down...")
        for name, proc in _bg_processes.items():
            try:
                proc.terminate()
                print(f"  Terminated {name}")
            except Exception:
                pass
        print("Done.")

---

## Usage Examples

Replace `https://your-url` with the URL printed in Step 7.

### curl

```bash
# Chat
curl -X POST https://your-url/api/chat \
  -H 'Content-Type: application/json' \
  -d '{"model": "qwen3:14b", "messages": [{"role": "user", "content": "Explain recursion"}]}'

# Generate (single-turn)
curl -X POST https://your-url/api/generate \
  -H 'Content-Type: application/json' \
  -d '{"model": "qwen3:14b", "prompt": "Hello, world!"}'

# Extended thinking (Qwen 3)
curl -X POST https://your-url/api/chat \
  -H 'Content-Type: application/json' \
  -d '{"model": "qwen3:14b", "messages": [{"role": "user", "content": "/think Solve step by step: ..."}]}'
```

### Python

```python
import requests

response = requests.post(
    "https://your-url/api/chat",
    json={
        "model": "qwen3:14b",
        "messages": [{"role": "user", "content": "Write a Flask API"}],
    }
)
print(response.json()["message"]["content"])
```

### VS Code (Continue / CodeGPT)

1. Install the extension
2. Set provider to **Ollama**
3. Set base URL to your tunnel URL
4. Select model (e.g. `qwen3:14b`)

---

## Notes

- **Quick tunnel**: URL changes every run — not suitable for persistent integrations.
- **ngrok free tier**: URL rotates every ~2 hours.
- **Cloudflare named tunnel**: Fixed hostname configured in your Cloudflare dashboard — best for long-running use.
- **Colab free tier**: Sessions last a few hours; daily GPU quota applies.
- **Colab Pro**: Sessions up to 24 hours with L4 GPU access.
- **Security**: The endpoint is public with no authentication. Avoid sending sensitive data.